# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Imports

In [ ]:
import os
import shutil
import traceback

import optuna
from optuna.trial import TrialState

from IPython.display import clear_output as clear

# Araras Optuna utilities
from araras.optuna.analysis.analyze import analyze_study
from araras.optuna.utils import (
    cleanup_non_top_trials,
    get_top_trials,
    init_study_dirs,
    rename_top_k_files,
    save_top_k_trials,
)

# Utility functions
from araras.utils.misc import (
    clear,
)

## 2. Run Parameters 

In [ ]:
TOP_K = 10 # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = True  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

In [ ]:
# Path to directory where the optuna_study dir is stored
STUDY_DIR = "/home/matheus/src/RayWise/results/v19/nas_cnn1d_flat_v19.2"

## Main

In [ ]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    # Initialize directories for the study
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
    ) = init_study_dirs(STUDY_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        direction="minimize",
        pruner=optuna.pruners.HyperbandPruner(),
        load_if_exists=True,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,
        rank_descending=RANK_DESCENDING,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # ————————————————————————————— Log Trial Results ———————————————————————————— #
    with open(f"{study_dir}/trials.log", "w") as f:
        f.write(
            f"Total trials: {len(study.trials)}\n"
            f"Pruned trials: {sum(t.state==TrialState.PRUNED for t in study.trials)}\n"
            f"Failed trials: {sum(t.state==TrialState.FAIL for t in study.trials)}\n"
        )

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (
        clear(),
        analyze_study(
            study,
            table_dir=os.path.join(study_dir, "analysis"),
            create_standalone=False,
            save_data=False,
            param_name_mapping=None,
        ),
    )
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)